In [20]:
# Задание 1

class Person:
    def __init__(self, first_name, last_name, gender):
        self.first_name = first_name
        self.last_name = last_name
        self.gender = gender  # 'M' или 'F'
        self.middle_name = None

    def set_middle_name(self, father):
        if father and father.gender == 'M':
            base = father.first_name
            if base[-1] == 'й' or base[-1] == 'ь':
                base = base[:-1] + 'е'
            else:
                base += 'о'
            self.middle_name = base + ('вич' if self.gender == 'M' else 'вна')

    def __repr__(self):
        return f"{self.last_name} {self.first_name} {self.middle_name or ''}".strip()


class Family:
    def __init__(self, father, mother):
        self.father = father
        self.mother = mother
        self.children = []

    def add_child(self, child):
        child.last_name = self.father.last_name
        child.set_middle_name(self.father)     
        self.children.append(child)

    def __repr__(self):
        members = [self.father, self.mother] + self.children
        return "Семья: " + ", ".join(str(m) for m in members)


class City:
    def __init__(self, name):
        self.name = name
        self.families = []

    def add_family(self, family):
        self.families.append(family)

    def show(self):
        print(f"Город {self.name}:")
        for f in self.families:
            print("  ", f)


father = Person("Андрей", "Петров", 'M')
mother = Person("Мария", "Сидорова", 'F')
child1 = Person("Алексей", "Петров", 'M')
child2 = Person("Ольга", "Петрова", 'F')

fam = Family(father, mother)
fam.add_child(child1)
fam.add_child(child2)

city = City("Москва")
city.add_family(fam)
city.show()

Город Москва:
   Семья: Петров Андрей, Сидорова Мария, Петров Алексей Андреевич, Петров Ольга Андреевна


In [21]:
# Задание 2

class Image:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.data = []
        
    def get_image(self):
        return self.data
    
    def get_width(self):
        return self.width
    
    def get_height(self):
        return self.height
        
        
class BinaryImage(Image):
    def __init__(self, width, height, bin_image):
        super().__init__(width, height)
        self.data = bin_image
        
class MonochromeImage(Image):
    def __init__(self, width, height, mono_image):
        super().__init__(width, height)
        self.data = mono_image
        
class ColouredImage(Image):
    def __init__(self, width, height, coloured_image):
        super().__init__(width, height)
        self.data = coloured_image
        
bn = BinaryImage(1, 1, [1])
bn.get_image()

mn = MonochromeImage(1, 1, [2])
mn.get_image()

mn = ColouredImage(1, 1, [3])
mn.get_image()


[3]

In [22]:
# Задание 3
import numpy as np
import cv2


class Converter:
    def monochrome_to_monochrome(src_img, trg_img):
        src = src_img.astype(np.float32)
        trg = trg_img.astype(np.float32)
        sm, ss = src.mean(), src.std()
        tm, ts = trg.mean(), trg.std()
        if ss == 0:
            ss = 1
        out = (src - sm) * (ts / ss) + tm
        return np.clip(out, 0, 255).astype(np.uint8)
    
    
    def coloured_to_coloured(src_img, trg_img):
        src_lab = cv2.cvtColor(src_img, cv2.COLOR_BGR2LAB).astype("float32")
        trg_lab = cv2.cvtColor(trg_img, cv2.COLOR_BGR2LAB).astype("float32")
        src_mean, src_std = cv2.meanStdDev(src_lab)
        trg_mean, trg_std = cv2.meanStdDev(trg_lab)
        src_mean, src_std = src_mean.flatten(), src_std.flatten()
        trg_mean, trg_std = trg_mean.flatten(), trg_std.flatten()
        src_std[src_std == 0] = 1
        out = np.zeros_like(src_lab, dtype=np.float32)
        for i in range(3):
            out[:, :, i] = (src_lab[:, :, i] - src_mean[i]) * (trg_std[i] / src_std[i]) + trg_mean[i]
        out = np.clip(out, 0, 255).astype(np.uint8)
        return cv2.cvtColor(out, cv2.COLOR_LAB2BGR)
    
    def binary_to_binary(img):
        return img
    
    def coloured_to_monochrome(img):
        return ((img[:, :, 0].astype(np.float32) + img[:, :, 1].astype(np.float32) + img[:, :, 2].astype(np.float32)) / 3.0).astype(np.uint8)
    
    def monochrome_to_coloured(img, palette=None):
        if palette is None:
            palette = cv2.applyColorMap(np.arange(256, dtype=np.uint8).reshape(256, 1), cv2.COLORMAP_JET).reshape(256, 3)
        return palette[img]

    
    def monochrome_to_binary(img, threshold=None):
        img = img.astype(np.uint8)
        if threshold is None:
            _, out = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            _, out = cv2.threshold(img, threshold, 255, cv2.THRESH_BINARY)
        return out
    
    def binary_to_monochrome(img):
        img = img.astype(np.uint8)
        inv = cv2.bitwise_not(img)
        dist = cv2.distanceTransform(inv, cv2.DIST_L2, 5)
        if dist.max() > 0:
            dist = dist / dist.max() * 255
        return dist.astype(np.uint8)
    
    def coloured_to_binary(img, threshold=None):
        mono = Converter.coloured_to_monochrome(img)
        return Converter.monochrome_to_binary(mono, threshold)

    def binary_to_coloured(img, palette=None):
        mono = Converter.binary_to_monochrome(img)
        return Converter.monochrome_to_coloured(mono, palette)